In [13]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_protocol import TextContentBlock
from rich import print as rprint

load_dotenv(override=True)

llm = init_chat_model(
    model="deepseek:deepseek-v4-flash"
)

## 1 PIIMiddleware

### 举例1：使用内置检测器

In [ ]:
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        PIIMiddleware(
            pii_type="email",
            strategy="redact",  # [REDACTED_EMAIL]
            apply_to_input=True
        ),
        PIIMiddleware(
            pii_type="credit_card",
            strategy="mask",  # ****-****-****-1234
            apply_to_input=True
        ),
        PIIMiddleware(
            pii_type="url",
            strategy="hash",  # a8f5f167...
            apply_to_input=True
        ),
        PIIMiddleware(
            pii_type="mac_address",
            strategy="mask",  # ****-****-****-1234
            apply_to_input=True
        ),
        PIIMiddleware(
            pii_type="ip",
            strategy="block",  # 触发 ValueError
            apply_to_input=True
        ),
    ]
)

resp = await agent.ainvoke({
    "messages":[
        HumanMessage(
            content="""
            帮我向 spf20031014@163.com 发送一封邮件
            同时帮我查看银行卡号： 5105-1051-0510-5100 的余额
            访问 https://localhost:12345
            确认这是不是MAC地址： 11-11-11-11-11-11
            """
        )
    ]
})

for msg in resp["messages"]:
    msg.pretty_print()

In [15]:
try:
    resp = await agent.ainvoke(
        {
            "messages":[
                HumanMessage(
                    content="看看这个ip 能不能ping通 192.168.0.1"
                )
            ]
        }
    )
except Exception as e:
    rprint(f"发生错误：{e}")
    rprint(type(e))

for msg in resp["messages"]:
    msg.pretty_print()


发生错误：Detected 1 instance(s) of ip in text content

<class 'langchain.agents.middleware._redaction.PIIDetectionError'>

================================ Human Message =================================

看看这个ip能不能ping通
================================== Ai Message ==================================

请把具体的 IP 地址发我。  
另外说明一下：我这边不能直接替你执行 `ping`，但可以告诉你怎么测、怎么看结果。

你可以本机运行：

- Windows：`ping IP地址`
- macOS/Linux：`ping -c 4 IP地址`
- 如果要测端口：Windows PowerShell 用  
  `Test-NetConnection IP地址 -Port 端口号`  
  Linux/macOS 用  
  `nc -vz IP地址 端口号`

把 IP 发来后，我可以告诉你对应命令和结果怎么判断。
